# Sleep Stage Signal Classification Preset

Reusable notebook for signal-processing hackathons where each prediction is a fixed-length time window.

This version is preset for the supplied sleep-stage archive:

- Train files: `83` continuous CSVs under `train/train/*.csv`
- Test segments: `7,832` CSV files under `test_segment/test_segment/<subject>/<id>.csv`
- Sampling rate: `16 Hz`
- Window length: `30 seconds = 480 rows`
- Signal columns: `BVP`, `ACC_X`, `ACC_Y`, `ACC_Z`, `TEMP`, `EDA`, `HR`, `IBI`
- Target column: `Sleep_Stage`
- Submission columns: `id`, `labels`
- Evaluation metric: weighted F1-score

The notebook is intentionally configurable: change the top config cell for other signal/window competitions.

In [ ]:
# ============================================================
# 0. Configuration
# ============================================================
from pathlib import Path
import zipfile
import os
import re
import time
import math
import json
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

RANDOM_STATE = 42

# ----- Dataset paths -----
# Works directly from zip, so you do not need to extract the 2GB archive.
ZIP_PATH = Path(r"super-ai-engineer-ss-6-sleep-stage-classification.zip")
EXTRACTED_DIR = None  # Example: Path("/kaggle/input/sleep-stage-classification")

TRAIN_GLOB_HINT = "train.csv"
TEST_GLOB_HINT = "test_segment.csv"
SAMPLE_SUB_NAME = "sample_submission.csv"

# ----- Competition schema -----
ID_COL = "id"
SUB_TARGET_COL = "labels"
TARGET_COL = "Sleep_Stage"

# ----- Signal/window settings -----
SAMPLE_RATE = 16
WINDOW_SECONDS = 30
WINDOW_SIZE = SAMPLE_RATE * WINDOW_SECONDS
SIGNAL_COLS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
ACC_COLS = ["ACC_X", "ACC_Y", "ACC_Z"]

# ----- Feature settings -----
ADD_ACC_MAGNITUDE = True
ADD_TIME_INDEX_FEATURES = True
ADD_PREV_NEXT_CONTEXT = True
FFT_COLS = ["BVP", "ACC_MAG", "EDA", "HR", "IBI"]
FFT_BANDS_HZ = [
    (0.00, 0.10),
    (0.10, 0.50),
    (0.50, 1.00),
    (1.00, 2.00),
    (2.00, 4.00),
    (4.00, 8.00),
]

# ----- Runtime controls -----
FORCE_REBUILD_FEATURES = False
FAST_MODE = False  # True processes fewer train/test files for quick notebook testing.
FAST_TRAIN_FILES = 8
FAST_TEST_FILES = 500
N_SPLITS = 5
USE_OPTIONAL_GBM = True

CACHE_DIR = Path("feature_cache_sleep_stage")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_FEATURES_PATH = CACHE_DIR / "train_features.csv"
TEST_FEATURES_PATH = CACHE_DIR / "test_features.csv"
OUTPUT_PATH = Path("submission.csv")

assert ZIP_PATH.exists() or EXTRACTED_DIR is not None, "Set ZIP_PATH or EXTRACTED_DIR."

In [ ]:
# ============================================================
# 1. File discovery and lightweight archive audit
# ============================================================
def list_files_from_zip(zip_path):
    zf = zipfile.ZipFile(zip_path)
    names = zf.namelist()
    train_files = sorted([n for n in names if n.startswith("train/") and n.endswith(".csv")])
    test_files = sorted([n for n in names if n.startswith("test_segment/") and n.endswith(".csv")])
    return zf, train_files, test_files


def list_files_from_dir(base_dir):
    base_dir = Path(base_dir)
    train_files = sorted(base_dir.glob(TRAIN_GLOB_HINT))
    test_files = sorted(base_dir.glob(TEST_GLOB_HINT))
    return None, train_files, test_files


def read_csv_any(handle, file_ref, **kwargs):
    if handle is None:
        return pd.read_csv(file_ref, **kwargs)
    return pd.read_csv(handle.open(file_ref), **kwargs)


if EXTRACTED_DIR is None:
    file_handle, train_files, test_files = list_files_from_zip(ZIP_PATH)
    sample_sub = pd.read_csv(file_handle.open(SAMPLE_SUB_NAME))
else:
    file_handle, train_files, test_files = list_files_from_dir(EXTRACTED_DIR)
    sample_sub = pd.read_csv(Path(EXTRACTED_DIR) / SAMPLE_SUB_NAME)

if FAST_MODE:
    train_files = train_files[:FAST_TRAIN_FILES]
    test_files = test_files[:FAST_TEST_FILES]

print("train files:", len(train_files))
print("test segment files:", len(test_files))
print("sample submission:", sample_sub.shape)
display(sample_sub.head())

preview_train = read_csv_any(file_handle, train_files[0], nrows=5)
preview_test = read_csv_any(file_handle, test_files[0], nrows=5)
print("train columns:", list(preview_train.columns))
print("test columns:", list(preview_test.columns))
display(preview_train)
display(preview_test)

In [ ]:
# ============================================================
# 2. Feature extraction helpers
# ============================================================
def subject_from_train_path(path_like):
    stem = Path(str(path_like)).stem
    return stem


def id_from_test_path(path_like):
    return Path(str(path_like)).stem


def subject_segment_from_id(segment_id):
    match = re.match(r"(.+?)_(\d+)$", str(segment_id))
    if match:
        return match.group(1), int(match.group(2))
    return str(segment_id).split("_")[0], -1


def safe_skew(x):
    x = np.asarray(x, dtype=float)
    m = np.nanmean(x)
    s = np.nanstd(x)
    if not np.isfinite(s) or s == 0:
        return 0.0
    return float(np.nanmean(((x - m) / s) ** 3))


def safe_kurtosis(x):
    x = np.asarray(x, dtype=float)
    m = np.nanmean(x)
    s = np.nanstd(x)
    if not np.isfinite(s) or s == 0:
        return 0.0
    return float(np.nanmean(((x - m) / s) ** 4) - 3.0)


def zero_cross_rate(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return 0.0
    centered = x - np.mean(x)
    return float(np.mean(np.diff(np.signbit(centered)) != 0))


def linear_slope(x):
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(x)
    if mask.sum() < 2:
        return 0.0
    idx = np.arange(len(x), dtype=float)[mask]
    vals = x[mask]
    idx = idx - idx.mean()
    denom = np.sum(idx ** 2)
    if denom == 0:
        return 0.0
    return float(np.sum(idx * (vals - vals.mean())) / denom)


def dominant_freq_and_bands(x, sample_rate, bands):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x, nan=np.nanmedian(x) if np.isfinite(x).any() else 0.0)
    if len(x) < 4:
        return {"dom_freq": 0.0, "spectral_entropy": 0.0, **{f"band_{lo:g}_{hi:g}": 0.0 for lo, hi in bands}}
    x = x - x.mean()
    freqs = np.fft.rfftfreq(len(x), d=1.0 / sample_rate)
    power = np.abs(np.fft.rfft(x)) ** 2
    total = float(power.sum()) + 1e-12
    nonzero = freqs > 0
    dom_freq = float(freqs[nonzero][np.argmax(power[nonzero])]) if np.any(nonzero) else 0.0
    p = power / total
    spectral_entropy = float(-np.sum(p * np.log(p + 1e-12)) / np.log(len(p) + 1e-12))
    out = {"dom_freq": dom_freq, "spectral_entropy": spectral_entropy}
    for lo, hi in bands:
        mask = (freqs >= lo) & (freqs < hi)
        out[f"band_{lo:g}_{hi:g}"] = float(power[mask].sum() / total)
    return out


def add_derived_signals(df):
    df = df.copy()
    if ADD_ACC_MAGNITUDE and all(c in df.columns for c in ACC_COLS):
        df["ACC_MAG"] = np.sqrt(np.sum(np.square(df[ACC_COLS].astype(float)), axis=1))
    return df


def extract_window_features(window_df):
    window_df = add_derived_signals(window_df)
    feature_cols = [c for c in window_df.columns if c != TARGET_COL and pd.api.types.is_numeric_dtype(window_df[c])]
    feats = {}
    for col in feature_cols:
        x = window_df[col].astype(float).values
        finite = x[np.isfinite(x)]
        if len(finite) == 0:
            finite = np.array([0.0])
        q05, q25, q50, q75, q95 = np.nanpercentile(finite, [5, 25, 50, 75, 95])
        dx = np.diff(np.nan_to_num(x, nan=np.nanmedian(finite)))
        prefix = col
        feats[f"{prefix}_mean"] = float(np.nanmean(finite))
        feats[f"{prefix}_std"] = float(np.nanstd(finite))
        feats[f"{prefix}_min"] = float(np.nanmin(finite))
        feats[f"{prefix}_max"] = float(np.nanmax(finite))
        feats[f"{prefix}_median"] = float(q50)
        feats[f"{prefix}_q05"] = float(q05)
        feats[f"{prefix}_q25"] = float(q25)
        feats[f"{prefix}_q75"] = float(q75)
        feats[f"{prefix}_q95"] = float(q95)
        feats[f"{prefix}_iqr"] = float(q75 - q25)
        feats[f"{prefix}_range"] = float(np.nanmax(finite) - np.nanmin(finite))
        feats[f"{prefix}_rms"] = float(np.sqrt(np.nanmean(finite ** 2)))
        feats[f"{prefix}_mad"] = float(np.nanmean(np.abs(finite - np.nanmean(finite))))
        feats[f"{prefix}_skew"] = safe_skew(finite)
        feats[f"{prefix}_kurtosis"] = safe_kurtosis(finite)
        feats[f"{prefix}_zcr"] = zero_cross_rate(finite)
        feats[f"{prefix}_slope"] = linear_slope(x)
        feats[f"{prefix}_diff_mean"] = float(np.nanmean(dx)) if len(dx) else 0.0
        feats[f"{prefix}_diff_std"] = float(np.nanstd(dx)) if len(dx) else 0.0
        feats[f"{prefix}_diff_abs_mean"] = float(np.nanmean(np.abs(dx))) if len(dx) else 0.0

        if col in FFT_COLS:
            fft_feats = dominant_freq_and_bands(x, SAMPLE_RATE, FFT_BANDS_HZ)
            for k, v in fft_feats.items():
                feats[f"{prefix}_{k}"] = v

    # Cross-axis accelerometer relationships.
    if all(c in window_df.columns for c in ACC_COLS):
        acc = window_df[ACC_COLS].astype(float)
        corr = acc.corr().fillna(0.0)
        feats["ACC_XY_corr"] = float(corr.loc["ACC_X", "ACC_Y"])
        feats["ACC_XZ_corr"] = float(corr.loc["ACC_X", "ACC_Z"])
        feats["ACC_YZ_corr"] = float(corr.loc["ACC_Y", "ACC_Z"])

    return feats

In [ ]:
# ============================================================
# 3. Build train/test feature tables with caching
# ============================================================
def make_train_features():
    rows = []
    label_counts = Counter()
    start = time.time()

    for file_i, file_ref in enumerate(train_files, 1):
        df = read_csv_any(file_handle, file_ref)
        missing_cols = [c for c in SIGNAL_COLS + [TARGET_COL] if c not in df.columns]
        if missing_cols:
            raise ValueError(f"{file_ref} is missing columns: {missing_cols}")

        subject = subject_from_train_path(file_ref)
        n_windows = len(df) // WINDOW_SIZE
        remainder = len(df) % WINDOW_SIZE
        if remainder:
            print(f"Warning: dropping {remainder} trailing rows from {file_ref}")

        for seg_idx in range(n_windows):
            lo = seg_idx * WINDOW_SIZE
            hi = lo + WINDOW_SIZE
            w = df.iloc[lo:hi]
            y = w[TARGET_COL].mode(dropna=True)
            label = y.iloc[0] if len(y) else np.nan
            label_counts[label] += 1
            feats = extract_window_features(w[SIGNAL_COLS + [TARGET_COL]])
            feats.update({
                ID_COL: f"{subject}_{seg_idx:05d}",
                "subject": subject,
                "segment_idx": seg_idx,
                TARGET_COL: label,
            })
            rows.append(feats)

        if file_i <= 3 or file_i % 10 == 0 or file_i == len(train_files):
            print(f"{file_i:03d}/{len(train_files)} {file_ref} -> {n_windows} windows; elapsed {time.time() - start:.1f}s")

    out = pd.DataFrame(rows)
    print("train features:", out.shape)
    print("label counts:", dict(label_counts))
    return out


def make_test_features():
    rows = []
    start = time.time()
    expected_ids = set(sample_sub[ID_COL].astype(str)) if ID_COL in sample_sub.columns else None

    for file_i, file_ref in enumerate(test_files, 1):
        df = read_csv_any(file_handle, file_ref)
        missing_cols = [c for c in SIGNAL_COLS if c not in df.columns]
        if missing_cols:
            raise ValueError(f"{file_ref} is missing columns: {missing_cols}")

        segment_id = id_from_test_path(file_ref)
        subject, seg_idx = subject_segment_from_id(segment_id)
        feats = extract_window_features(df[SIGNAL_COLS])
        feats.update({
            ID_COL: segment_id,
            "subject": subject,
            "segment_idx": seg_idx,
        })
        rows.append(feats)

        if file_i <= 3 or file_i % 1000 == 0 or file_i == len(test_files):
            print(f"{file_i:05d}/{len(test_files)} {file_ref}; elapsed {time.time() - start:.1f}s")

    out = pd.DataFrame(rows)
    if expected_ids is not None:
        missing = expected_ids - set(out[ID_COL].astype(str))
        extra = set(out[ID_COL].astype(str)) - expected_ids
        print("test ids missing from features:", len(missing))
        print("feature ids not in submission:", len(extra))
    print("test features:", out.shape)
    return out


if FORCE_REBUILD_FEATURES or not TRAIN_FEATURES_PATH.exists():
    train_feat = make_train_features()
    train_feat.to_csv(TRAIN_FEATURES_PATH, index=False)
else:
    train_feat = pd.read_csv(TRAIN_FEATURES_PATH)

if FORCE_REBUILD_FEATURES or not TEST_FEATURES_PATH.exists():
    test_feat = make_test_features()
    test_feat.to_csv(TEST_FEATURES_PATH, index=False)
else:
    test_feat = pd.read_csv(TEST_FEATURES_PATH)

print("Loaded train features:", train_feat.shape)
print("Loaded test features:", test_feat.shape)
display(train_feat.head())

In [ ]:
# ============================================================
# 4. Add optional sequence context features
# ============================================================
def add_context_features(df, feature_cols):
    df = df.sort_values(["subject", "segment_idx"]).reset_index(drop=True).copy()
    if not ADD_TIME_INDEX_FEATURES:
        return df

    group = df.groupby("subject", sort=False)
    max_idx = group["segment_idx"].transform("max").replace(0, 1)
    df["segment_pos_norm"] = df["segment_idx"] / max_idx
    df["subject_n_segments"] = group["segment_idx"].transform("count")

    if ADD_PREV_NEXT_CONTEXT:
        # Keep this narrow by adding context only to stable low-frequency summary features.
        context_base = [
            c for c in feature_cols
            if c.endswith("_mean") or c.endswith("_std") or c.endswith("_median")
        ]
        context_base = context_base[:80]
        for c in context_base:
            df[f"{c}_prev"] = group[c].shift(1)
            df[f"{c}_next"] = group[c].shift(-1)
            df[f"{c}_roll3_mean"] = group[c].transform(lambda s: s.rolling(3, min_periods=1, center=True).mean())
    return df


base_feature_cols = [c for c in train_feat.columns if c not in [ID_COL, TARGET_COL, "subject"]]
train_feat = add_context_features(train_feat, base_feature_cols)
test_feat = add_context_features(test_feat, [c for c in test_feat.columns if c not in [ID_COL, "subject"]])

print("after context:", train_feat.shape, test_feat.shape)

In [ ]:
# ============================================================
# 5. Prepare modeling matrices
# ============================================================
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier

train_feat = train_feat.dropna(subset=[TARGET_COL]).reset_index(drop=True)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_feat[TARGET_COL])
class_names = list(label_encoder.classes_)
print("classes:", dict(enumerate(class_names)))
display(train_feat[TARGET_COL].value_counts().to_frame("windows"))

DROP_COLS = [ID_COL, TARGET_COL, "subject"]
feature_cols = [c for c in train_feat.columns if c not in DROP_COLS]
feature_cols = [c for c in feature_cols if c in test_feat.columns]

X = train_feat[feature_cols].replace([np.inf, -np.inf], np.nan)
X_test = test_feat[feature_cols].replace([np.inf, -np.inf], np.nan)
groups = train_feat["subject"].astype(str).values

print("X:", X.shape)
print("X_test:", X_test.shape)
print("subjects:", pd.Series(groups).nunique())

In [ ]:
# ============================================================
# 6. Model zoo for weighted-F1 classification
# ============================================================
models = {
    "logreg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            C=2.0,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
    "extra_trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=300 if FAST_MODE else 900,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
    "hist_gbdt": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=150 if FAST_MODE else 500,
            l2_regularization=0.03,
            random_state=RANDOM_STATE,
        )),
    ]),
}

if USE_OPTIONAL_GBM:
    try:
        from lightgbm import LGBMClassifier
        models["lightgbm"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                objective="multiclass",
                n_estimators=500 if FAST_MODE else 1800,
                learning_rate=0.03,
                num_leaves=63,
                subsample=0.9,
                colsample_bytree=0.85,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])
    except Exception as e:
        print("LightGBM skipped:", type(e).__name__)

    try:
        from catboost import CatBoostClassifier
        models["catboost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostClassifier(
                iterations=400 if FAST_MODE else 1400,
                learning_rate=0.035,
                depth=6,
                loss_function="MultiClass",
                auto_class_weights="Balanced",
                random_seed=RANDOM_STATE,
                verbose=False,
            )),
        ])
    except Exception as e:
        print("CatBoost skipped:", type(e).__name__)

print("models:", list(models))

In [ ]:
# ============================================================
# 7. Grouped cross-validation scored by weighted F1
# ============================================================
def make_splitter():
    n_subjects = pd.Series(groups).nunique()
    if n_subjects >= N_SPLITS:
        return StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE), groups
    return StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE), None


splitter, split_groups = make_splitter()
cv_rows = []
oof_proba_store = {}

for model_name, model in models.items():
    print(f"\n=== {model_name} ===")
    oof_proba = np.zeros((len(X), len(class_names)), dtype=float)
    fold_scores = []

    split_iter = splitter.split(X, y, split_groups) if split_groups is not None else splitter.split(X, y)
    for fold, (tr_idx, va_idx) in enumerate(split_iter, 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        model.fit(X_tr, y_tr)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_va)
            oof_proba[va_idx, :proba.shape[1]] = proba
            pred = np.argmax(proba, axis=1)
        else:
            pred = model.predict(X_va)
            oof_proba[va_idx, pred] = 1.0

        weighted_f1 = f1_score(y_va, pred, average="weighted")
        macro_f1 = f1_score(y_va, pred, average="macro")
        fold_scores.append(weighted_f1)
        cv_rows.append({
            "model": model_name,
            "fold": fold,
            "weighted_f1": weighted_f1,
            "macro_f1": macro_f1,
        })
        print(f"fold {fold}: weighted_f1={weighted_f1:.5f}, macro_f1={macro_f1:.5f}")

    oof_pred = np.argmax(oof_proba, axis=1)
    print("OOF weighted_f1:", f1_score(y, oof_pred, average="weighted"))
    print("OOF macro_f1:", f1_score(y, oof_pred, average="macro"))
    oof_proba_store[model_name] = oof_proba

cv = pd.DataFrame(cv_rows)
summary = cv.groupby("model").mean(numeric_only=True).sort_values("weighted_f1", ascending=False)
display(summary)

In [ ]:
# ============================================================
# 8. Blend, inspect OOF report, and fit final models
# ============================================================
TOP_MODELS = summary.index.tolist()[: min(3, len(summary))]
print("Selected models:", TOP_MODELS)

blend_oof = np.mean([oof_proba_store[m] for m in TOP_MODELS], axis=0)
blend_pred = np.argmax(blend_oof, axis=1)

print("Blend weighted F1:", f1_score(y, blend_pred, average="weighted"))
print("Blend macro F1:", f1_score(y, blend_pred, average="macro"))
print(classification_report(y, blend_pred, target_names=class_names))
display(pd.DataFrame(confusion_matrix(y, blend_pred), index=class_names, columns=class_names))

fitted_models = {}
test_probas = []

for model_name in TOP_MODELS:
    print("fit full:", model_name)
    model = models[model_name]
    model.fit(X, y)
    fitted_models[model_name] = model
    proba = model.predict_proba(X_test)
    test_probas.append(proba)

blend_test = np.mean(test_probas, axis=0)
test_pred = np.argmax(blend_test, axis=1)
test_labels = label_encoder.inverse_transform(test_pred)

In [ ]:
# ============================================================
# 9. Submission
# ============================================================
submission = sample_sub.copy()

if ID_COL not in submission.columns:
    submission[ID_COL] = test_feat[ID_COL].values

pred_df = pd.DataFrame({
    ID_COL: test_feat[ID_COL].astype(str).values,
    SUB_TARGET_COL: test_labels,
})

# Preserve sample submission row order.
submission[ID_COL] = submission[ID_COL].astype(str)
submission = submission[[ID_COL]].merge(pred_df, on=ID_COL, how="left")

if submission[SUB_TARGET_COL].isna().any():
    missing = submission[submission[SUB_TARGET_COL].isna()][ID_COL].head(10).tolist()
    raise ValueError(f"Missing predictions for IDs like: {missing}")

submission.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH.resolve())
display(submission.head())
display(submission[SUB_TARGET_COL].value_counts().to_frame("predicted_count"))

In [ ]:
# ============================================================
# 10. Feature importance
# ============================================================
importance_frames = []

for model_name, pipe in fitted_models.items():
    estimator = pipe.named_steps.get("model")
    if hasattr(estimator, "feature_importances_"):
        importance_frames.append(pd.DataFrame({
            "model": model_name,
            "feature": feature_cols,
            "importance": estimator.feature_importances_,
        }))
    elif hasattr(estimator, "coef_"):
        coef = np.asarray(estimator.coef_)
        importance_frames.append(pd.DataFrame({
            "model": model_name,
            "feature": feature_cols,
            "importance": np.mean(np.abs(coef), axis=0),
        }))

if importance_frames:
    importance = pd.concat(importance_frames, ignore_index=True)
    display(
        importance.groupby("feature")["importance"]
        .mean()
        .sort_values(ascending=False)
        .head(50)
        .to_frame()
    )
else:
    print("No built-in feature importance available.")

## Customization checklist

For another signal-processing hackathon, edit the config cell first:

1. Set `ZIP_PATH` or `EXTRACTED_DIR`.
2. Set `SIGNAL_COLS`, `TARGET_COL`, `ID_COL`, and `SUB_TARGET_COL`.
3. Set `SAMPLE_RATE`, `WINDOW_SECONDS`, and `WINDOW_SIZE`.
4. If train files are already segmented, change `make_train_features()` to read one label per file instead of slicing continuous files.
5. Edit `FFT_BANDS_HZ` for the physiology or sensor domain.
6. Use grouped CV whenever samples from the same subject/session/device can leak across folds.
7. Rank models by the actual competition metric. For this hackathon, it is weighted F1.

Good next upgrades:

- Add subject-level normalization before feature extraction.
- Add wavelet or bandpass features if SciPy/PyWavelets are available.
- Add transition smoothing for consecutive test segments from the same subject.
- Train a 1D CNN/Transformer only after the feature baseline is stable.